# Fase 2 do CRISP-DM — Compreensão dos Dados

**Trabalho N1 — Ciência de Dados** · Centro Universitário Católica de Santa Catarina
Prof. Dr. Claudinei Dias (Ney) · 2026

**Ferramenta de BI:** Plotly
**Dataset:** Hotel Booking Demand (119.390 reservas × 32 atributos)

**Integrantes:** Henrique Cordeiro de Oliveira · Lucas Mendonça · Victor Kunz · Nicholas Scoz · Kauã Lucindo

---

## Objetivo de negócio

> Reduzir a taxa de cancelamento de reservas, identificando quais características
> da reserva e do cliente mais se associam ao cancelamento — para que a rede
> hoteleira possa antecipar cancelamentos e agir sobre eles (política de depósito,
> overbooking calibrado, contato proativo).

Este notebook cobre a **Fase 2 (Compreensão dos Dados)**. A limpeza e a preparação
ficam no notebook `02_preparacao_dos_dados.ipynb`.

## 1. Configuração do ambiente

O Colab já traz pandas e plotly. Fixamos versões compatíveis entre `plotly` e
`kaleido` porque é a dupla que exporta os gráficos como PNG para os slides.

In [ ]:
!pip install -q -U "plotly>=6.0" "kaleido>=1.0"

In [ ]:
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Sem esta linha os gráficos podem sair em branco ao reabrir o notebook no Colab.
pio.renderers.default = "colab"

print("pandas", pd.__version__, "| plotly", plotly.__version__)

### Identidade visual dos gráficos

Definir a paleta uma vez, num `template` do Plotly, evita que cada gráfico saia de
uma cor diferente — os slides ficam visualmente coesos.

As duas cores de série foram validadas para **daltonismo** (ΔE 24,7 entre elas na
simulação de protanopia, contra um piso de 8) e ambas têm contraste ≥ 3:1 sobre o
fundo claro. A escala da matriz de correlação é **divergente**: dois polos de cor
com cinza neutro no meio, para que "correlação zero" não pareça um valor qualquer.

In [ ]:
# --- Superfície e tinta ---
SUPERFICIE       = "#fcfcfb"
TEXTO_PRIMARIO   = "#0b0b0b"
TEXTO_SECUNDARIO = "#52514e"
GRADE            = "#e5e4e0"

# --- Séries categóricas (ordem fixa, nunca ciclada) ---
SERIE_1 = "#2a78d6"   # azul
SERIE_2 = "#eb6834"   # laranja

# --- Escala divergente para correlação (polo / neutro / polo) ---
ESCALA_DIVERGENTE = [
    [0.00, "#8f2020"],
    [0.25, "#d03b3b"],
    [0.50, "#f0efec"],
    [0.75, "#2a78d6"],
    [1.00, "#104281"],
]

pio.templates["n1"] = go.layout.Template(
    layout=dict(
        font=dict(family="Inter, Segoe UI, sans-serif", size=13, color=TEXTO_PRIMARIO),
        title=dict(font=dict(size=17), x=0, xanchor="left"),
        paper_bgcolor=SUPERFICIE,
        plot_bgcolor=SUPERFICIE,
        colorway=[SERIE_1, SERIE_2],
        xaxis=dict(showgrid=False, linecolor=GRADE, ticks="outside", tickcolor=GRADE,
                   title=dict(font=dict(color=TEXTO_SECUNDARIO))),
        yaxis=dict(gridcolor=GRADE, zeroline=False, linecolor=GRADE,
                   title=dict(font=dict(color=TEXTO_SECUNDARIO))),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title=dict(text="")),
        margin=dict(t=70, r=30, b=60, l=70),
    )
)
pio.templates.default = "n1"

## 2. Extração — carregando o dataset

O arquivo vive no repositório do trabalho, então o notebook roda no Colab sem
nenhum upload manual: basta abrir e executar.

Os valores ausentes vêm gravados como a string `"NULL"`. O pandas já reconhece esse
texto como nulo por padrão e o converte para `NaN` — vale conferir isso na prática,
porque um ausente que passa despercebido vira erro silencioso na Fase 3.

In [ ]:
URL = ("https://raw.githubusercontent.com/mendoncaluucas/DataScience/"
       "main/data/raw/hotel_bookings.csv")

df = pd.read_csv(URL)

print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas".replace(",", "."))
df.head()

## 3. Primeira inspeção — tipos e memória

`info()` responde de uma vez: quantas colunas são numéricas, quantas são texto, e
quais têm menos registros preenchidos que o total (sinal de ausentes).

In [ ]:
df.info()

## 4. Profiling — mapa de valores ausentes

Equivale ao painel "Qualidade da Coluna" do Power Query, que o enunciado cita: antes
de decidir qualquer tratamento, é preciso enxergar **onde** e **quanto** falta.

In [ ]:
ausentes = (df.isna().sum()
              .loc[lambda s: s > 0]
              .sort_values(ascending=False)
              .rename("qtd_ausente")
              .to_frame())
ausentes["pct_ausente"] = (ausentes["qtd_ausente"] / len(df) * 100).round(2)
ausentes

In [ ]:
fig_ausentes = px.bar(
    ausentes.reset_index(names="coluna"),
    x="pct_ausente", y="coluna", orientation="h", text="pct_ausente",
    title="Valores ausentes por coluna",
    labels={"pct_ausente": "% dos registros", "coluna": ""},
)
fig_ausentes.update_traces(
    marker_color=SERIE_1, marker_cornerradius=4,
    texttemplate="%{text:.1f}%", textposition="outside",
    textfont_color=TEXTO_SECUNDARIO,
    hovertemplate="<b>%{y}</b><br>%{x:.2f}% ausente<extra></extra>",
)
fig_ausentes.update_layout(yaxis=dict(categoryorder="total ascending"),
                           xaxis=dict(range=[0, 105]), height=320)
fig_ausentes.show()

**O que isso já decide para a Fase 3** — cada nível de ausência pede uma resposta
diferente, e é essa justificativa que o trabalho cobra:

| Coluna | Ausente | Encaminhamento |
|---|---|---|
| `company` | ~94% | Forte candidata a **remoção** — quase não há informação |
| `agent` | ~14% | Preencher com categoria explícita ("sem agência") |
| `country` | ~0,4% | Preencher com a moda ou marcar como "desconhecido" |
| `children` | 4 registros | Preencher com 0 |

## 5. Estatísticas descritivas

In [ ]:
df.describe().T.round(2)

## 6. Análise exploratória

O enunciado pede **2 a 3 gráficos** que revelem insights iniciais. Os três abaixo
atacam o objetivo de negócio por ângulos diferentes: *onde* cancela mais, *quando* a
reserva foi feita, e *o que* se move junto com o cancelamento.

### 6.1 Onde se cancela mais?

In [ ]:
taxa = (df.groupby("hotel")["is_canceled"]
          .mean().mul(100).round(1)
          .reset_index(name="taxa_cancelamento"))

fig_hotel = px.bar(
    taxa, x="hotel", y="taxa_cancelamento", text="taxa_cancelamento",
    title="Taxa de cancelamento por tipo de hotel",
    labels={"hotel": "", "taxa_cancelamento": "Reservas canceladas (%)"},
)
fig_hotel.update_traces(
    marker_color=SERIE_1, marker_cornerradius=4, width=0.45,
    texttemplate="%{text:.1f}%", textposition="outside",
    textfont_color=TEXTO_SECUNDARIO,
    hovertemplate="<b>%{x}</b><br>%{y:.1f}% canceladas<extra></extra>",
)
fig_hotel.update_layout(yaxis=dict(range=[0, 50]), height=380)
fig_hotel.show()

### 6.2 Reservas feitas com muita antecedência cancelam mais?

In [ ]:
df_situacao = df.assign(
    situacao=df["is_canceled"].map({0: "Mantida", 1: "Cancelada"})
)

fig_lead = px.box(
    df_situacao, x="situacao", y="lead_time", color="situacao",
    category_orders={"situacao": ["Mantida", "Cancelada"]},
    color_discrete_map={"Mantida": SERIE_1, "Cancelada": SERIE_2},
    title="Antecedência da reserva (lead time) por situação",
    labels={"situacao": "", "lead_time": "Dias entre a reserva e o check-in"},
)
fig_lead.update_traces(marker=dict(opacity=0.3, size=4), line=dict(width=2))
# O eixo X já nomeia os dois grupos — a legenda seria pura repetição.
fig_lead.update_layout(height=420, showlegend=False)
fig_lead.show()

### 6.3 O que se move junto com o cancelamento?

Matriz de correlação restrita às variáveis numéricas mais relevantes — as 18
numéricas do dataset inteiro deixariam os rótulos ilegíveis.

In [ ]:
# Rótulos curtos em português: os nomes originais estouram a margem e ficam
# ilegíveis na projeção — e a plateia não precisa decorar o nome técnico da coluna.
ROTULOS = {
    "is_canceled":                 "Cancelou",
    "lead_time":                   "Antecedência",
    "adr":                         "Diária média",
    "total_of_special_requests":   "Pedidos especiais",
    "booking_changes":             "Alterações",
    "previous_cancellations":      "Cancel. anteriores",
    "required_car_parking_spaces": "Vagas de garagem",
    "stays_in_week_nights":        "Noites (semana)",
    "stays_in_weekend_nights":     "Noites (fim de semana)",
    "adults":                      "Adultos",
}

corr = (df[list(ROTULOS)].corr()
          .round(2)
          .rename(index=ROTULOS, columns=ROTULOS))

fig_corr = px.imshow(
    corr, text_auto=True, aspect="auto",
    color_continuous_scale=ESCALA_DIVERGENTE, zmin=-1, zmax=1,
    title="Correlação entre variáveis numéricas",
)
fig_corr.update_layout(
    height=620,
    coloraxis_colorbar=dict(title="r", thickness=14),
    margin=dict(l=170, b=150, t=70, r=30),   # espaço para os rótulos longos
)
fig_corr.update_xaxes(tickangle=-45)
fig_corr.show()

## 7. Hipóteses

O enunciado pede **2 hipóteses** formuladas a partir da análise acima. Os rascunhos
abaixo foram escritos a partir do que os gráficos devem mostrar — **confiram contra
os números que saírem na execução de vocês** e ajustem a redação.

---

**Hipótese 1 —** *Reservas feitas com maior antecedência têm maior probabilidade de
cancelamento.* A mediana de `lead_time` das reservas canceladas deve ficar
visivelmente acima da mediana das mantidas, e `lead_time` deve ser a variável com
maior correlação positiva com `is_canceled`.

> Por que importa para o negócio: se confirmada, reservas antigas viram uma fila de
> risco que pode receber confirmação proativa.

**Hipótese 2 —** *Hóspedes que fazem pedidos especiais cancelam menos.* Espera-se
correlação **negativa** entre `total_of_special_requests` e `is_canceled` — o pedido
especial funcionaria como sinal de compromisso com a viagem.

> Por que importa para o negócio: dá um indicador barato de "reserva firme", sem
> precisar de modelo preditivo.

---

*(Se algum gráfico contrariar o rascunho, reescrevam a hipótese — hipótese refutada e
documentada vale mais que hipótese ajustada depois do resultado.)*

## 8. Exportar os gráficos para os slides

O GitHub **não renderiza gráficos Plotly** no preview de notebook — são JavaScript, e
o preview é estático. Por isso salvamos PNG: são os *prints* que o enunciado pede na
Parte II.

Se a exportação falhar, o plano B é o ícone de câmera na barra de cada gráfico.

In [ ]:
from pathlib import Path

PASTA_PRINTS = Path("prints")
PASTA_PRINTS.mkdir(exist_ok=True)

graficos = {
    "01_valores_ausentes":       fig_ausentes,
    "02_cancelamento_por_hotel": fig_hotel,
    "03_lead_time_por_situacao": fig_lead,
    "04_correlacao":             fig_corr,
}

for nome, figura in graficos.items():
    try:
        # Respeitar a altura de cada gráfico. Forçar um tamanho único estica as
        # barras e deforma a proporção pensada para cada um.
        figura.write_image(PASTA_PRINTS / f"{nome}.png",
                           width=1200, height=figura.layout.height, scale=2)
        print(f"ok      {nome}.png")
    except Exception as erro:
        print(f"falhou  {nome}.png -> {type(erro).__name__}: {erro}")
        print("        plano B: ícone de câmera na barra do gráfico")

In [ ]:
# Baixa todos os prints de uma vez (só funciona no Colab)
import shutil
from google.colab import files

shutil.make_archive("prints_n1", "zip", PASTA_PRINTS)
files.download("prints_n1.zip")

## 9. Versão offline para o seminário

Esta célula gera **um único arquivo HTML** com os quatro gráficos, totalmente
interativos e com a biblioteca Plotly embutida dentro dele.

Por que isso importa numa apresentação ao vivo:

- **Não depende de internet, de login ou do Colab.** Abre com dois cliques em
  qualquer navegador, em qualquer computador — inclusive no da sala.
- **Mantém a interatividade**, que é justamente o que diferencia o Plotly de uma
  biblioteca de gráfico estático. Hover, zoom, seleção e legenda clicável continuam
  funcionando.
- É o **plano B** se o Wi-Fi cair ou a sessão do Colab desconectar no meio da
  demonstração.

O arquivo sai com cerca de 4 MB, porque carrega o `plotly.js` inteiro dentro de si.
É esse peso que o torna independente de rede.

In [ ]:
from pathlib import Path

ARQUIVO_OFFLINE = Path("demonstracao_n1.html")

# include_plotlyjs=True apenas no primeiro gráfico: embute a biblioteca uma vez
# e os demais reaproveitam. Com "cdn" o arquivo ficaria leve, mas exigiria internet.
partes = []
for i, (nome, figura) in enumerate(graficos.items()):
    partes.append(figura.to_html(full_html=False,
                                 include_plotlyjs=(i == 0),
                                 default_width="100%"))

corpo = "".join(f'<section>{p}</section>' for p in partes)

html = f"""<!doctype html>
<html lang="pt-br">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Fase 2 do CRISP-DM — Compreensão dos Dados</title>
<style>
  body {{ margin:0; padding:32px 20px; background:{SUPERFICIE};
         color:{TEXTO_PRIMARIO}; font-family:Inter,"Segoe UI",sans-serif; }}
  header, section {{ max-width:1100px; margin-left:auto; margin-right:auto; }}
  header {{ margin-bottom:28px; border-bottom:1px solid {GRADE}; padding-bottom:18px; }}
  h1 {{ font-size:21px; margin:0 0 6px; font-weight:600; }}
  p.sub {{ color:{TEXTO_SECUNDARIO}; margin:0; font-size:14px; line-height:1.5; }}
  section {{ margin-bottom:44px; }}
</style>
</head>
<body>
<header>
  <h1>Fase 2 do CRISP-DM — Compreensão dos Dados</h1>
  <p class="sub">
    Trabalho N1 · Ciência de Dados · Centro Universitário Católica SC<br>
    Ferramenta: Plotly · Dataset: Hotel Booking Demand (119.390 reservas × 32 atributos)
  </p>
</header>
{corpo}
</body>
</html>"""

ARQUIVO_OFFLINE.write_text(html, encoding="utf-8")
tamanho = ARQUIVO_OFFLINE.stat().st_size / 1_000_000
print(f"{ARQUIVO_OFFLINE} gerado ({tamanho:.1f} MB)")
print("Baixe o arquivo e teste ANTES do seminário, com o Wi-Fi desligado.")

In [ ]:
# Baixa a versão offline (só funciona no Colab)
from google.colab import files

files.download(str(ARQUIVO_OFFLINE))

---

## Próximo passo

`02_preparacao_dos_dados.ipynb` — **Fase 3 do CRISP-DM**, onde entram os três itens
obrigatórios do enunciado:

1. Estratégia de tratamento de valores ausentes
2. Criação de uma nova coluna (engenharia de feature)
3. Remoção de uma coluna irrelevante, com justificativa